# Step 40 — Build separate hypocenter source indexes (including former step 41)

This notebook embeds all three ingestion workflows: the HYPO71 summary adapter, original PINAALL parser and Randy White annotated PINAALL parser. Shared FLOVOpy readers remain imported. The parsers use distinct function/constant prefixes so that running one section cannot replace another parser's helpers.

No source records are merged here. Each run replaces its own configured CSV/log outputs under the external output root; legacy files are unchanged. Definitions and configuration cells do not run ingestion. Run the summary and selected PINAALL sections below, then inspect logs before steps 42–44. No production ingestion was executed as part of this notebook consolidation.

## Sources and interpretation

| Source | Implementation | Output / caution |
|---|---|---|
| `Pinatubo_all.sum` | FLOVOpy VDAP HYPO71 line reader with the existing CSV adapter | Core origin/location/depth/magnitude and available quality fields, source line and raw text. “Summary” is not a declaration that this is the authoritative final solution. |
| Original `PINAALL.DAT` | Embedded tolerant token parser | Original step-41 CSV path; raw lines and unparsed reasons retained. Timestamp representation does not independently establish UTC. |
| Annotated Randy White PINAALL | Embedded annotated parser | Separate origin/annotation/error/audit outputs; reported time, inferred time basis and notes retained. Do not silently replace the original index downstream. |

Depth is stored in km and coordinates in decimal degrees. Magnitude and optional numeric fields retain the parser's interpretation; confirm their meanings from source documentation before scientific use. Neither parser relocates earthquakes or establishes network response calibration.

## 1. Shared configuration

Inspect the resolved source/output roots. Each source section has separate settings and run cells.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

def preview_result(result):
    for name, value in result.items():
        print(name)
        display(value.head() if hasattr(value, 'head') else
                value[:5] if isinstance(value, list) else value)

Configuration: /Users/thompsong/Developer/Pinatubo1991SeismicData/10_conversion/conversion_config.json
External outputs: /Users/thompsong/work/Pinatubo1991Processing/converted


## 2. HYPO71 summary implementation

The adapter calls `flovopy.core.vdap.parse_hypo71_line` and retains the existing step-40 schema. Arithmetic time rollover is handled by that shared reader. Blank lines are skipped; nonblank records the reader cannot parse are written to the unparsed-line log. The file is read with decoding errors ignored, so raw text is a textual provenance aid rather than a byte-perfect copy. Preserve the original file for forensic review.

The parser writes an empty table with headers when no records parse. Inspect row counts and the log rather than treating CSV existence as success. Quality columns `nass` and `rms` retain reader values; their presence does not impose a quality filter.

In [2]:
from pathlib import Path
from typing import Optional, Dict, List

import pandas as pd
from flovopy.core.vdap import parse_hypo71_line as _parse_vdap_hypo71_line

run_summary_PARSER_ID = "vdap_hypo71_adapter_v2"


# ============================================================================
# HYPO71-LIKE LINE PARSER
# ============================================================================

def run_summary_parse_hypo71_line(line):
    """Adapt the authoritative VDAP parser to the unchanged Step-40 CSV schema."""
    value = _parse_vdap_hypo71_line(line)
    if value is None:
        return None
    return dict(origin_time=value['origin_time'].isoformat(),latitude=value['latitude'],
        longitude=value['longitude'],depth_km=value['depth_km'],magnitude=value['magnitude'],
        nass=value['n_ass_raw'],rms=value['time_residual_raw'])



# ============================================================================
# MAIN
# ============================================================================

def run_summary(
    *,
    summary_file,
    out_csv,
    error_log
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        summary_file=summary_file,
        out_csv=out_csv,
        error_log=error_log,
    )

    src = Path(args.summary_file)
    if not src.exists():
        raise SystemExit(f"File not found: {src}")

    rows = []
    unparsed: List[str] = []

    with src.open("r", errors="ignore") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.rstrip("\n")
            if not line.strip():
                continue

            parsed = run_summary_parse_hypo71_line(line)
            if parsed is None:
                unparsed.append(f"{lineno}: {raw}")
                continue

            parsed.update({
                "source_file": src.name,
                "source_line": lineno,
                "parser": run_summary_PARSER_ID,
                "raw_line": line,
            })
            rows.append(parsed)

    # --- Write outputs ---
    out_csv = Path(args.out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=["origin_time", "latitude", "longitude", "depth_km", "magnitude", "nass", "rms", "source_file", "source_line", "parser", "raw_line"]).to_csv(out_csv, index=False)

    err_path = Path(args.error_log)
    err_path.parent.mkdir(parents=True, exist_ok=True)
    with err_path.open("w") as f:
        for l in unparsed:
            f.write(l if l.endswith("\n") else l + "\n")

    # --- Summary ---
    print("\nSTEP 40 — HYPOCENTER INDEX BUILT")
    print("--------------------------------")
    print(f"Source file:     {src}")
    print(f"Parsed rows:     {len(rows)}")
    print(f"Unparsed lines:  {len(unparsed)}")
    print(f"Index CSV:       {out_csv}")
    print(f"Error log:       {err_path}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## HYPO71 summary — Pinatubo_all.sum

This produces `40_summary_file_index.csv` and the unparsed-line log. It does not depend on the PINAALL section below.


In [3]:
SUMMARY_SETTINGS = dict(
    summary_file=P['SUMMARY_FILE_40'],
    out_csv=P['SUMMARY_FILE_40_INDEX'],
    error_log=P['SUMMARY_FILE_40_ERR'],
)
display(SUMMARY_SETTINGS)


{'summary_file': '/Users/thompsong/Library/CloudStorage/Box-Box/thompsong/DATA/pinatubo1991/LEGACY/EVENT_METADATA/HYPOCENTERS/Pinatubo_all.sum',
 'out_csv': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/40_summary_file_index.csv',
 'error_log': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/40_summary_file_unparsed_lines.txt'}

In [4]:
summary_result = run_summary(**SUMMARY_SETTINGS)



STEP 40 — HYPOCENTER INDEX BUILT
--------------------------------
Source file:     /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/DATA/pinatubo1991/LEGACY/EVENT_METADATA/HYPOCENTERS/Pinatubo_all.sum
Parsed rows:     4632
Unparsed lines:  0
Index CSV:       /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/40_summary_file_index.csv
Error log:       /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/40_summary_file_unparsed_lines.txt


In [5]:
print('Summary index:', SUMMARY_SETTINGS['out_csv'])
preview_result(summary_result)


Summary index: /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/40_summary_file_index.csv
rows


[{'origin_time': '1991-05-07T03:21:14.290000',
  'latitude': 15.168666666666667,
  'longitude': 120.32233333333333,
  'depth_km': 3.28,
  'magnitude': 1.36,
  'nass': '7',
  'rms': '0.10',
  'source_file': 'Pinatubo_all.sum',
  'source_line': 1,
  'parser': 'vdap_hypo71_adapter_v2',
  'raw_line': '91 5 7  321 14.29 15n10.12 120E19.34   3.28   1.36  7          0.10'},
 {'origin_time': '1991-05-07T12:52:29.950000',
  'latitude': 15.169833333333333,
  'longitude': 120.3135,
  'depth_km': 1.89,
  'magnitude': 0.7,
  'nass': '7',
  'rms': '0.10',
  'source_file': 'Pinatubo_all.sum',
  'source_line': 2,
  'parser': 'vdap_hypo71_adapter_v2',
  'raw_line': '91 5 7 1252 29.95 15n10.19 120E18.81   1.89   0.70  7          0.10'},
 {'origin_time': '1991-05-08T02:37:00.060000',
  'latitude': 15.163333333333334,
  'longitude': 120.33116666666666,
  'depth_km': 4.55,
  'magnitude': 1.0,
  'nass': '7',
  'rms': '0.08',
  'source_file': 'Pinatubo_all.sum',
  'source_line': 3,
  'parser': 'vdap_hypo71_a

unparsed


[]

## 3. Original PINAALL implementation

This parser handles variable date/time token spacing and arithmetic minute/second rollover. It retains source filenames/line numbers, raw text and parser identification. Review unparsed reasons and unusual locations/depths separately; arithmetic normalization cannot prove a malformed record's intended time. The original parser labels times in its existing UTC-style representation without consulting the annotated catalogue's clock history. Do not interpret agreement of timestamp formats as proof of shared clocks.

In [6]:
from __future__ import annotations

import re
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
from obspy import UTCDateTime

run_original_PARSER_ID = "pinaall_dat_v3"


# =============================================================================
# Helpers: tolerant date/time parsing
# =============================================================================

@dataclass
class run_original_DTParseResult:
    dt: datetime
    next_index: int


def run_original__split_mmdd(tok: str) -> Optional[Tuple[int, int]]:
    if not tok.isdigit():
        return None
    if len(tok) == 3:
        m = int(tok[0])
        d = int(tok[1:])
    elif len(tok) == 4:
        m = int(tok[:2])
        d = int(tok[2:])
    else:
        return None
    if 1 <= m <= 12 and 1 <= d <= 31:
        return m, d
    return None


def run_original__parse_hhmm(tok: str) -> Optional[Tuple[int, int]]:
    if not tok.isdigit():
        return None
    v = int(tok)
    if len(tok) in (3, 4):
        h = v // 100
        mi = v % 100
        if 0 <= h <= 23 and 0 <= mi <= 99:
            return h, mi
    return None


def run_original__normalize_minute_overflow(h: int, mi: int, sec: float, base: datetime) -> datetime:
    extra_h, mi2 = divmod(mi, 60)
    dt0 = base.replace(hour=h, minute=0, second=0, microsecond=0)
    return dt0 + timedelta(hours=extra_h, minutes=mi2, seconds=sec)


def run_original__looks_like_number(s: str) -> bool:
    try:
        float(s)
        return True
    except Exception:
        return False


def run_original_parse_datetime_tokens(tokens: List[str], i0: int = 0) -> run_original_DTParseResult:
    i = i0
    if i + 3 >= len(tokens):
        raise ValueError("Not enough tokens for datetime")

    yy = int(tokens[i]); i += 1
    year = 1900 + yy if yy < 100 else yy

    md = run_original__split_mmdd(tokens[i])
    if md is not None:
        month, day = md
        i += 1
    else:
        month = int(tokens[i])
        day = int(tokens[i + 1])
        i += 2

    if (
        i + 2 < len(tokens)
        and tokens[i].isdigit()
        and tokens[i + 1].isdigit()
        and run_original__looks_like_number(tokens[i + 2])
    ):
        h = int(tokens[i])
        mi = int(tokens[i + 1])
        sec = float(tokens[i + 2])
        i += 3
    else:
        hm = run_original__parse_hhmm(tokens[i])
        if hm is None:
            raise ValueError(f"Bad HHMM token: {tokens[i]!r}")
        h, mi = hm
        sec = float(tokens[i + 1])
        i += 2

    base = datetime(year, month, day, 0, 0, 0, tzinfo=timezone.utc)
    dt = run_original__normalize_minute_overflow(h, mi, sec, base)
    return run_original_DTParseResult(dt=dt, next_index=i)


# =============================================================================
# Helpers: tolerant lat/lon parsing
# =============================================================================

run_original_LATLON_ONE_TOKEN_RE = re.compile(r"^(?P<deg>\d+)(?P<hem>[NnSsEeWw])(?P<min>\d+(\.\d+)?)$")
run_original_DEG_HEM_RE = re.compile(r"^(?P<deg>\d+)(?P<hem>[NnSsEeWw])$")
run_original_HEM_ONLY_RE = re.compile(r"^(?P<hem>[NnSsEeWw])$")


def run_original__hem_sign(hem: str) -> int:
    return -1 if hem.upper() in ("S", "W") else 1


def run_original_parse_latlon_from_tokens(tokens: List[str], i: int) -> Tuple[float, int, str]:
    t0 = tokens[i]

    m = run_original_LATLON_ONE_TOKEN_RE.match(t0)
    if m:
        deg = float(m.group("deg"))
        mins = float(m.group("min"))
        hem = m.group("hem")
        val = run_original__hem_sign(hem) * (deg + mins / 60.0)
        return val, i + 1, t0

    m = run_original_DEG_HEM_RE.match(t0)
    if m:
        mins = float(tokens[i + 1])
        deg = float(m.group("deg"))
        hem = m.group("hem")
        val = run_original__hem_sign(hem) * (deg + mins / 60.0)
        return val, i + 2, f"{t0} {tokens[i+1]}"

    if t0.isdigit():
        hem = tokens[i + 1]
        mins = float(tokens[i + 2])
        val = run_original__hem_sign(hem) * (float(t0) + mins / 60.0)
        return val, i + 3, f"{t0} {hem} {mins}"

    raise ValueError(f"Cannot parse lat/lon at token {i}")


def run_original_parse_depth_mag_from_tokens(tokens: List[str], i: int) -> Tuple[Optional[float], Optional[float], int]:
    nums: List[float] = []
    j = i
    while j < len(tokens) and len(nums) < 2:
        if run_original__looks_like_number(tokens[j]):
            nums.append(float(tokens[j]))
        j += 1

    depth = nums[0] if len(nums) >= 1 else None
    mag = nums[1] if len(nums) >= 2 else None
    return depth, mag, j


# =============================================================================
# Line parser
# =============================================================================

def run_original_parse_pinaall_line(line: str) -> Tuple[Optional[Dict[str, object]], Optional[str]]:
    tokens = line.strip().split()
    if len(tokens) < 6:
        return None, "too_few_tokens"

    try:
        dt_res = run_original_parse_datetime_tokens(tokens)
        dt = dt_res.dt
        i = dt_res.next_index
    except Exception as e:
        return None, f"datetime_failed: {e}"

    try:
        lat, i, _ = run_original_parse_latlon_from_tokens(tokens, i)
        lon, i, _ = run_original_parse_latlon_from_tokens(tokens, i)
    except Exception as e:
        return None, f"latlon_failed: {e}"

    depth_km, mag, _ = run_original_parse_depth_mag_from_tokens(tokens, i)

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return None, "latlon_out_of_range"

    origin_time = UTCDateTime(dt).isoformat()

    return {
        "origin_time": origin_time,
        "latitude": lat,
        "longitude": lon,
        "depth_km": depth_km,
        "magnitude": mag,
    }, None


# =============================================================================
# Main
# =============================================================================

def run_original(
    *,
    pinaall_file,
    out_csv,
    error_log
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        pinaall_file=pinaall_file,
        out_csv=out_csv,
        error_log=error_log,
    )

    src = Path(args.pinaall_file)
    rows: List[Dict[str, object]] = []
    unparsed: List[str] = []

    with src.open("r", errors="ignore") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.rstrip("\n")
            if not line.strip():
                continue

            parsed, err = run_original_parse_pinaall_line(line)
            if parsed is None:
                unparsed.append(f"{lineno}: {err}: {raw}")
                continue

            parsed.update({
                "source_file": src.name,
                "source_line": lineno,
                "parser": run_original_PARSER_ID,
                "raw_line": line,
            })
            rows.append(parsed)

    out_csv = Path(args.out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_csv, index=False)

    err_path = Path(args.error_log)
    err_path.parent.mkdir(parents=True, exist_ok=True)
    err_path.write_text("\n".join(unparsed))

    print("\nSTEP 41 — PINAALL.DAT HYPOCENTER INDEX BUILT")
    print("-------------------------------------------")
    print(f"Source file:     {src}")
    print(f"Parsed rows:     {len(rows)}")
    print(f"Unparsed lines:  {len(unparsed)}")
    print(f"Index CSV:       {out_csv}")
    print(f"Error log:       {err_path}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 4. Annotated PINAALL implementation and explicit clock policy

This alternative preserves inline/standalone annotations and reported timestamps. Its classification rules are existing parser assumptions attributed in that code to Randy's October 2026 correspondence; confidence labels such as CONFIRMED are inherited classifications, not new independent verification:

| Reported interval (1991) | Existing classifier |
|---|---|
| Before May 20 | LOCAL, inferred |
| May 20 through May 21 before 02:00 | LOCAL |
| May 21 02:00–before 11:00 | UNCERTAIN |
| May 21 11:00 through May 23 | LOCAL_PLUS_1MIN |
| May 24–26 | LOCAL |
| May 27–29 | UNCERTAIN |
| May 30 onward | UTC |

`LOCAL_UTC_OFFSET=None` now correctly leaves local-origin UTC fields blank; it does **not** subtract eight hours. With an explicit offset of 8, LOCAL subtracts eight hours and LOCAL_PLUS_1MIN also subtracts one minute (recorded clock assumed one minute fast). UNCERTAIN records remain unresolved; UTC records retain their reported time. This fixes the previous hard-coded offset of 8 inside `parse_catalog` while preserving reported timestamps and classification rules.

These intervals apply to this particular catalogue interpretation, not every computer/data product. Check the source correspondence before adopting corrections. There are no automatic CVO Pacific-time corrections. Cross-data-type alignment still requires evidence.

The second numeric field after longitude remains mapped to magnitude for compatibility; its meaning is not established here. Narrative dates without coordinates can be stored as annotations rather than parse errors. Error counts alone therefore do not measure rejected earthquake records. Downstream association must exclude unresolved times explicitly, preserving them for review; never coerce blank UTC fields into a guessed timezone.

In [7]:
from __future__ import annotations

import csv
import re

from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path


run_annotated_PARSER_ID = "pinaall_annotated_rw_v1"

run_annotated_DEFAULT_SOURCE = "PINAALL_annotated_RW_20261013.DAT"

# Randy's stated transition date.
run_annotated_TIME_TRANSITION = datetime(1991, 5, 30)

# Latitude/longitude coordinates in degrees and minutes.
#
# Examples:
#   15n 9.34 120e19.56
#   15n26.44 120e22.83
#
run_annotated_COORD_RE = re.compile(
    r"(?P<latdeg>\d{1,2})\s*"
    r"(?P<lathem>[NSns])\s*"
    r"(?P<latmin>\d+(?:\.\d+)?)"
    r"\s+"
    r"(?P<londeg>\d{1,3})\s*"
    r"(?P<lonhem>[EWew])\s*"
    r"(?P<lonmin>\d+(?:\.\d+)?)"
)

run_annotated_EVENT_PREFIX_RE = re.compile(r"^\s*91\s+")

run_annotated_NUMBER_RE = re.compile(
    r"^[+-]?(?:\d+(?:\.\d*)?|\.\d+)$"
)


# ---------------------------------------------------------------------
# Date and time parsing
# ---------------------------------------------------------------------

def run_annotated_parse_month_day(tokens):
    """
    Parse either:

        510       -> May 10
        0510      -> May 10
        5 10      -> May 10

    Returns
    -------
    month, day, next_index
    """

    token = tokens[1]

    if len(token) in (3, 4) and token.isdigit():

        month = int(token[:-2])
        day = int(token[-2:])

        if not (1 <= month <= 12 and 1 <= day <= 31):
            raise ValueError("Invalid month/day")

        return month, day, 2

    month = int(tokens[1])
    day = int(tokens[2])

    if not (1 <= month <= 12 and 1 <= day <= 31):
        raise ValueError("Invalid month/day")

    return month, day, 3


def run_annotated_parse_time_tokens(tokens):
    """
    Parse the date/time portion preceding latitude.

    Examples
    --------
    91 510  410 52.99
    91 512  2 4 25.20
    91 521 11 7 46.96
    91 818  032 24.05

    Returns
    -------
    datetime
        Naive datetime preserving the reported clock time.
    """

    if len(tokens) < 4:
        raise ValueError("Too few date/time tokens")

    yy = int(tokens[0])

    year = 1900 + yy if yy >= 70 else 2000 + yy

    month, day, i = run_annotated_parse_month_day(tokens)

    remaining = tokens[i:]

    if len(remaining) == 2:

        hhmm = remaining[0]

        if not hhmm.isdigit():
            raise ValueError("Invalid HHMM")

        value = int(hhmm)

        hour = value // 100
        minute = value % 100

        seconds = float(remaining[1])

    elif len(remaining) == 3:

        hour = int(remaining[0])
        minute = int(remaining[1])
        seconds = float(remaining[2])

    else:
        raise ValueError(
            f"Unexpected time fields: {remaining}"
        )

    if not 0 <= hour <= 23:
        raise ValueError(f"Invalid hour: {hour}")

    if not 0 <= minute <= 99:
        raise ValueError(f"Invalid minute: {minute}")

    if not 0 <= seconds < 120:
        raise ValueError(f"Invalid seconds: {seconds}")

    # Timedelta handles minute/second rollover.
    base = datetime(year, month, day)

    return base + timedelta(
        hours=hour,
        minutes=minute,
        seconds=seconds
    )


# ---------------------------------------------------------------------
# Coordinate parsing
# ---------------------------------------------------------------------

def run_annotated_parse_coordinates(match):
    """
    Convert degrees/minutes to decimal degrees.
    """

    latdeg = float(match.group("latdeg"))
    latmin = float(match.group("latmin"))

    londeg = float(match.group("londeg"))
    lonmin = float(match.group("lonmin"))

    if not 0 <= latmin < 60:
        raise ValueError("Latitude minutes out of range")

    if not 0 <= lonmin < 60:
        raise ValueError("Longitude minutes out of range")

    lat = latdeg + latmin / 60.0
    lon = londeg + lonmin / 60.0

    if match.group("lathem").upper() == "S":
        lat = -lat

    if match.group("lonhem").upper() == "W":
        lon = -lon

    if not (-90 <= lat <= 90):
        raise ValueError("Latitude out of range")

    if not (-180 <= lon <= 180):
        raise ValueError("Longitude out of range")

    return lat, lon


# ---------------------------------------------------------------------
# Time convention
# ---------------------------------------------------------------------

def run_annotated_classify_time_basis(dt):
    """
    Classify Pinatubo catalog time conventions.

    Based on Randy White's email of October 2026.

    Returns
    -------
    basis : str
        LOCAL, LOCAL_PLUS_1MIN, UTC, or UNCERTAIN

    confidence : str
        CONFIRMED, INFERRED, or UNKNOWN

    note : str
        Explanation of the classification.
    """

    if dt < datetime(1991, 5, 20):

        return (
            "LOCAL",
            "INFERRED",
            "Pre-May-20 records; local time assumed."
        )

    if dt < datetime(1991, 5, 21, 2):

        return (
            "LOCAL",
            "CONFIRMED",
            "May 20 to May 21 02:00: local time."
        )

    if dt < datetime(1991, 5, 21, 11):

        return (
            "UNCERTAIN",
            "UNKNOWN",
            "May 21 02:00–11:00 transition interval."
        )

    if dt < datetime(1991, 5, 24):

        return (
            "LOCAL_PLUS_1MIN",
            "CONFIRMED",
            "May 21 11:00 through May 23."
        )

    if dt < datetime(1991, 5, 27):

        return (
            "LOCAL",
            "CONFIRMED",
            "May 24–26: local time."
        )

    if dt < datetime(1991, 5, 30):

        return (
            "UNCERTAIN",
            "UNKNOWN",
            "May 27–29: time convention unresolved."
        )

    return (
        "UTC",
        "CONFIRMED",
        "May 30 onward: GMT."
    )


def run_annotated_convert_origin_time(dt, basis, local_utc_offset=8):
    """
    Convert reported catalog time to UTC.

    Philippine local time = UTC + 8 hours.

    LOCAL_PLUS_1MIN means the recorded clock is
    one minute ahead of Philippine local time.
    """

    if basis == "UTC":

        utc_dt = dt

    elif basis == "LOCAL":

        utc_dt = dt - timedelta(
            hours=local_utc_offset
        )

    elif basis == "LOCAL_PLUS_1MIN":

        utc_dt = dt - timedelta(
            hours=local_utc_offset,
            minutes=1
        )

    else:

        return ""

    return (
        utc_dt.replace(tzinfo=timezone.utc)
        .isoformat()
        .replace("+00:00", "Z")
    )
# ---------------------------------------------------------------------
# Numeric fields and annotations
# ---------------------------------------------------------------------

def run_annotated_split_numeric_and_annotations(text):
    """
    Extract leading numeric fields and remaining annotation.

    Supports:
        3.37 0.93 D
        4.20 0.38!
        3.17 0.66!merged
        1.66 0.00!p
        1.90 0.00! Pvt
        8.29 1.18 6 0.14

    Returns
    -------
    numbers : list[float]
        Numeric fields preceding the first annotation.

    annotation : str
        Annotation text, including attached punctuation.
    """

    tokens = text.strip().split()

    numbers = []
    annotation_tokens = []

    # Match a numeric prefix, optionally followed by annotation text.
    numeric_prefix = re.compile(
        r"^([+-]?(?:\d+(?:\.\d*)?|\.\d+))(.*)$"
    )

    in_annotation = False

    for token in tokens:

        if in_annotation:
            annotation_tokens.append(token)
            continue

        match = numeric_prefix.match(token)

        if match is None:
            in_annotation = True
            annotation_tokens.append(token)
            continue

        numbers.append(float(match.group(1)))

        suffix = match.group(2)

        if suffix:
            in_annotation = True
            annotation_tokens.append(suffix)

    return numbers, " ".join(annotation_tokens)

def run_annotated_classify_event(annotation):
    """
    Classify events using Randy White's annotation codes.

    Recognized codes
    ----------------
    D       : distal
    DVT     : distal volcano-tectonic
    p       : proximal
    SB      : spasmodic burst
    SBurst  : spasmodic burst

    Returns
    -------
    str
        Normalized event classification, or an empty
        string if no recognized classification is found.
    """

    import re

    tokens = re.findall(r"[A-Za-z]+", annotation)

    # More specific classifications first.
    for token in tokens:
        if token.upper() == "DVT":
            return "distal_vt"

    for token in tokens:
        if token.upper() in ("SB", "SBURST"):
            return "spasmodic_burst"

    for token in tokens:
        if token == "D":
            return "distal"

        if token == "p":
            return "proximal"

    return ""

# ---------------------------------------------------------------------
# Earthquake record parser
# ---------------------------------------------------------------------

def run_annotated_parse_event_line(line):
    """
    Parse a single hypocenter record.

    Returns
    -------
    dict
        Parsed fields.

    Raises
    ------
    ValueError
        When a candidate event cannot be parsed.
    """

    match = run_annotated_COORD_RE.search(line)

    if match is None:
        raise ValueError("No valid coordinate fields")

    datetime_text = line[:match.start()].strip()

    dt = run_annotated_parse_time_tokens(datetime_text.split())

    lat, lon = run_annotated_parse_coordinates(match)

    tail = line[match.end():]

    numbers, annotation = run_annotated_split_numeric_and_annotations(tail)

    if len(numbers) < 2:
        raise ValueError(
            "Expected at least depth and second numeric field"
        )

    depth_km = numbers[0]
    magnitude = numbers[1]

    extra_numbers = numbers[2:]

    return {
        "reported_time": dt,
        "latitude": lat,
        "longitude": lon,
        "depth_km": depth_km,
        "magnitude": magnitude,
        "extra_numeric_fields": extra_numbers,
        "annotation": annotation,
    }


# ---------------------------------------------------------------------
# Catalog processing
# ---------------------------------------------------------------------

def run_annotated_parse_catalog(
    source,
    local_utc_offset=None,
):
    """
    Parse complete annotated catalog.

    Returns
    -------
    events, annotations, errors
    """

    events = []
    annotations = []
    errors = []

    current_context = []

    lines = source.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for lineno, line in enumerate(lines, start=1):

        stripped = line.strip()

        if not stripped:
            continue

        # Standalone comments and annotations.
        if not run_annotated_EVENT_PREFIX_RE.match(line):

            annotations.append({
                "source_line": lineno,
                "annotation_type": "standalone",
                "text": stripped,
                "raw_line": line,
            })

            current_context.append(
                f"L{lineno}: {stripped}"
            )

            continue

        try:

            parsed = run_annotated_parse_event_line(line)

        except ValueError as exc:

            # Date-prefixed narrative annotations, such as
            # "91 615 CATACLYSMIC ERUPTION STARTS",
            # are preserved separately.
            if run_annotated_COORD_RE.search(line) is None:

                annotations.append({
                    "source_line": lineno,
                    "annotation_type": "dated_annotation",
                    "text": stripped,
                    "raw_line": line,
                })

                current_context.append(
                    f"L{lineno}: {stripped}"
                )

            else:

                errors.append({
                    "source_line": lineno,
                    "reason": str(exc),
                    "raw_line": line,
                })

            continue

        dt = parsed["reported_time"]

        basis, confidence, time_note = run_annotated_classify_time_basis(dt)

        origin_time = run_annotated_convert_origin_time(
            dt,
            basis,
            local_utc_offset=local_utc_offset
        ) if basis == "UTC" or local_utc_offset is not None else ""

        extra = parsed["extra_numeric_fields"]
        event_class = run_annotated_classify_event(parsed["annotation"])

        event = {
            "origin_time": origin_time,
            "reported_time": dt.isoformat(),
            "time_basis": basis,
            "time_confidence": confidence,
            "time_note": time_note, 
            "latitude": parsed["latitude"],
            "longitude": parsed["longitude"],
            "depth_km": parsed["depth_km"],
            "magnitude": parsed["magnitude"],
            "event_class": event_class,

            # Preserve optional numeric columns.
            "extra_numeric_1": (
                extra[0] if len(extra) > 0 else ""
            ),
            "extra_numeric_2": (
                extra[1] if len(extra) > 1 else ""
            ),
            "extra_numeric_remaining": (
                " ".join(map(str, extra[2:]))
                if len(extra) > 2 else ""
            ),

            "annotation": parsed["annotation"],

            # Standalone annotations since the preceding event.
            "preceding_annotations": " | ".join(
                current_context
            ),

            "source_file": source.name,
            "source_line": lineno,
            "parser": run_annotated_PARSER_ID,
            "raw_line": line,
        }

        events.append(event)

        if parsed["annotation"]:

            annotations.append({
                "source_line": lineno,
                "annotation_type": "inline",
                "text": parsed["annotation"],
                "raw_line": line,
            })

        # Comments apply to the next successfully parsed event.
        current_context = []

    return events, annotations, errors


# ---------------------------------------------------------------------
# CSV output
# ---------------------------------------------------------------------

run_annotated_EVENT_COLUMNS = [
    "origin_time",
    "reported_time",
    "time_basis",
    "time_confidence",
    "time_note",
    "latitude",
    "longitude",
    "depth_km",
    "magnitude",
    "extra_numeric_1",
    "extra_numeric_2",
    "extra_numeric_remaining",
    "annotation",
    "event_class",
    "preceding_annotations",
    "source_file",
    "source_line",
    "parser",
    "raw_line",
]

run_annotated_ANNOTATION_COLUMNS = [
    "source_line",
    "annotation_type",
    "text",
    "raw_line",
]

run_annotated_ERROR_COLUMNS = [
    "source_line",
    "reason",
    "raw_line",
]


def run_annotated_write_csv(path, rows, columns):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with path.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=columns
        )

        writer.writeheader()

        writer.writerows(rows)


# ---------------------------------------------------------------------
# Audit report
# ---------------------------------------------------------------------

def run_annotated_write_audit(
    path,
    source,
    events,
    annotations,
    errors,
    local_utc_offset,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    basis_counts = Counter(
        e["time_basis"] for e in events
    )

    annotation_counts = Counter(
        a["annotation_type"] for a in annotations
    )

    event_class_counts = Counter(
        e["event_class"] or "unclassified"
        for e in events
    )

    reported_times = sorted(
        e["reported_time"] for e in events
    )

    depths = [
        e["depth_km"] for e in events
    ]

    latitudes = [
        e["latitude"] for e in events
    ]

    longitudes = [
        e["longitude"] for e in events
    ]

    lines = [
        "STEP 41 — PINATUBO HYPOCENTER AUDIT",
        "=" * 45,
        "",
        f"Parser: {run_annotated_PARSER_ID}",
        f"Source: {source}",
        "",
        f"Parsed earthquakes: {len(events)}",
        f"Annotations: {len(annotations)}",
        f"Parsing errors: {len(errors)}",
        "",
        "Time basis:",
    ]

    for basis, count in sorted(basis_counts.items()):
        lines.append(f"  {basis}: {count}")

    lines.extend([
        "",
        f"Local UTC offset: {local_utc_offset}",
        "",
        "Annotation types:",
    ])

    for kind, count in sorted(annotation_counts.items()):
        lines.append(f"  {kind}: {count}")

    lines.extend([
        "",
        "Event classifications:",
    ])

    for event_class, count in sorted(event_class_counts.items()):
        lines.append(f"  {event_class}: {count}")

    if events:

        lines.extend([
            "",
            "Catalog ranges:",
            f"  First reported time: {reported_times[0]}",
            f"  Last reported time:  {reported_times[-1]}",
            f"  Latitude:  {min(latitudes):.5f} "
            f"to {max(latitudes):.5f}",
            f"  Longitude: {min(longitudes):.5f} "
            f"to {max(longitudes):.5f}",
            f"  Depth:     {min(depths):.2f} "
            f"to {max(depths):.2f} km",
        ])

    if errors:

        lines.extend([
            "",
            "Parsing errors:",
        ])

        for error in errors:

            lines.append(
                f"  Line {error['source_line']}: "
                f"{error['reason']}"
            )

    path.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8"
    )


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

def run_annotated(
    *,
    source=run_annotated_DEFAULT_SOURCE,
    out_csv='pinatubo_hypocenters.csv',
    annotations_csv='pinatubo_annotations.csv',
    error_log='pinatubo_parse_errors.csv',
    audit_report='pinatubo_parse_summary.txt',
    local_utc_offset=None
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        source=source,
        out_csv=out_csv,
        annotations_csv=annotations_csv,
        error_log=error_log,
        audit_report=audit_report,
        local_utc_offset=local_utc_offset,
    )

    source = Path(args.source)

    if not source.exists():
        raise ValueError(f"File not found: {source}")

    events, annotations, errors = run_annotated_parse_catalog(
        source,
        local_utc_offset=args.local_utc_offset,
    )

    run_annotated_write_csv(
        Path(args.out_csv),
        events,
        run_annotated_EVENT_COLUMNS
    )

    run_annotated_write_csv(
        Path(args.annotations_csv),
        annotations,
        run_annotated_ANNOTATION_COLUMNS
    )

    run_annotated_write_csv(
        Path(args.error_log),
        errors,
        run_annotated_ERROR_COLUMNS
    )

    run_annotated_write_audit(
        Path(args.audit_report),
        source,
        events,
        annotations,
        errors,
        args.local_utc_offset,
    )

    print()
    print("STEP 41 — PINATUBO HYPOCENTER INDEX")
    print("----------------------------------")
    print(f"Source:       {source}")
    print(f"Earthquakes:  {len(events)}")
    print(f"Annotations:  {len(annotations)}")
    print(f"Errors:       {len(errors)}")
    print()
    print(f"Catalog:      {args.out_csv}")
    print(f"Annotations:  {args.annotations_csv}")
    print(f"Errors:       {args.error_log}")
    print(f"Audit:        {args.audit_report}")
    print()
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## PINAALL — original or annotated parser

Choose `PARSER = "original"` (default) or `"annotated"`, then rerun this section's settings cell. The original parser writes `41_summary_file_index.csv`. The annotated parser writes `41_annotated_summary_file_index.csv`, plus annotation, error and audit files.

`LOCAL_UTC_OFFSET = None` leaves unresolved local-time records unresolved. Use `8.0` only when adopting the documented Philippine UTC+08 conversion. Inspect annotated time-basis and transition-date warnings before UTC association. The second numeric field is retained as magnitude for compatibility, but its interpretation needs confirmation.

This section can run independently of the summary-file section. Selecting annotated input does not replace the canonical step-41 index used downstream.


In [8]:
PARSER = 'original'  # Choose 'original' or 'annotated'.
LOCAL_UTC_OFFSET = None  # Annotated parser only; 8.0 means Philippine UTC+08.

if PARSER == 'original':
    run_pinaall = run_original
    PINAALL_SETTINGS = dict(
        pinaall_file=P['SUMMARY_FILE_41'],
        out_csv=P['SUMMARY_FILE_41_INDEX'],
        error_log=P['SUMMARY_FILE_41_ERR'],
    )
elif PARSER == 'annotated':
    run_pinaall = run_annotated
    PINAALL_SETTINGS = dict(
        source=str(Path(P['LEGACY_HYPO_DIR']) / 'PINAALL_annotated_RW_20261013.DAT'),
        out_csv=str(Path(P['TMP_DIR']) / '41_annotated_summary_file_index.csv'),
        annotations_csv=str(Path(P['TMP_DIR']) / '41_annotations.csv'),
        error_log=str(Path(P['TMP_DIR']) / '41_annotated_parse_errors.csv'),
        audit_report=str(Path(P['TMP_DIR']) / '41_annotated_parse_summary.txt'),
        local_utc_offset=LOCAL_UTC_OFFSET,
    )
else:
    raise ValueError("PARSER must be 'original' or 'annotated'")

print('Selected PINAALL parser:', PARSER)
display(PINAALL_SETTINGS)


Selected PINAALL parser: original


{'pinaall_file': '/Users/thompsong/Library/CloudStorage/Box-Box/thompsong/DATA/pinatubo1991/LEGACY/EVENT_METADATA/HYPOCENTERS/PINAALL.DAT',
 'out_csv': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/41_summary_file_index.csv',
 'error_log': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/41_summary_file_unparsed_lines.txt'}

In [9]:
pinaall_result = run_pinaall(**PINAALL_SETTINGS)



STEP 41 — PINAALL.DAT HYPOCENTER INDEX BUILT
-------------------------------------------
Source file:     /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/DATA/pinatubo1991/LEGACY/EVENT_METADATA/HYPOCENTERS/PINAALL.DAT
Parsed rows:     3601
Unparsed lines:  0
Index CSV:       /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/41_summary_file_index.csv
Error log:       /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/41_summary_file_unparsed_lines.txt


In [10]:
print('PINAALL parser:', PARSER)
print('PINAALL index:', PINAALL_SETTINGS['out_csv'])
preview_result(pinaall_result)


PINAALL parser: original
PINAALL index: /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/41_summary_file_index.csv
rows


[{'origin_time': '1991-05-10T04:10:52.990000',
  'latitude': 15.440666666666667,
  'longitude': 120.3805,
  'depth_km': 15.0,
  'magnitude': 0.0,
  'source_file': 'PINAALL.DAT',
  'source_line': 1,
  'parser': 'pinaall_dat_v3',
  'raw_line': '91 510  410 52.99 15n26.44 120e22.83  15.00   0.00'},
 {'origin_time': '1991-05-10T05:15:26.520000',
  'latitude': 15.155666666666667,
  'longitude': 120.326,
  'depth_km': 3.37,
  'magnitude': 0.93,
  'source_file': 'PINAALL.DAT',
  'source_line': 2,
  'parser': 'pinaall_dat_v3',
  'raw_line': '91 510  515 26.52 15n 9.34 120e19.56   3.37   0.93'},
 {'origin_time': '1991-05-11T21:18:53.740000',
  'latitude': 15.152833333333334,
  'longitude': 120.32183333333333,
  'depth_km': 2.53,
  'magnitude': 0.88,
  'source_file': 'PINAALL.DAT',
  'source_line': 3,
  'parser': 'pinaall_dat_v3',
  'raw_line': '91 511 2118 53.74 15n 9.17 120e19.31   2.53   0.88'},
 {'origin_time': '1991-05-12T02:04:25.200000',
  'latitude': 15.166833333333333,
  'longitude': 12

unparsed


[]

## Continue with comparison and association

Steps 42–44 still consume the original `40_summary_file_index.csv` and `41_summary_file_index.csv` paths by default. If you chose annotated PINAALL, set their `hypo41` input to `PINAALL_SETTINGS['out_csv']` after reviewing time-basis flags.

The source records remain in separate tables; step 43 performs the later association. Review both parsers' error logs before continuing. Former source-specific notebooks are preserved under `superseded/hypocenter_ingestion`.
